# KneeXrayData HDF5 to production-YOLO ROI dataset

This is a **dataset audit and deterministic ROI builder**, not a training notebook. It uses the original Mendeley/OAI HDF5 files, which contain one bilateral radiograph, two ground-truth boxes, two KL grades, and the patient ID per case.

The current production YOLOv8 detector creates the actual ROI crops. Ground-truth boxes are used only to validate/match detections and attach the correct right/left KL label. The generated ROI images are raw integer `xyxy` crops: no CLAHE, square padding, resizing, mirroring, or augmentation is baked into the dataset. The notebook also exports the HDF5 bilateral source images as standalone `full_images/<split>/<patient>.png` files at their stored `256x320` resolution.

Run all cells in order with a Colab T4 GPU. Do not use `auto_test`: it overlaps the published test patients. The notebook never changes the downloaded source dataset.

In [ ]:
import subprocess
import sys

subprocess.check_call([
    sys.executable,
    "-m",
    "pip",
    "install",
    "-q",
    "ultralytics>=8.1,<9",
    "h5py>=3.9",
    "pandas>=2.0",
    "seaborn>=0.13",
])
print("Dependencies installed. A GPU runtime is recommended for the YOLO cell.")


In [ ]:
from google.colab import drive
drive.mount("/content/drive", force_remount=False)

import hashlib
import itertools
import json
import math
import re
import shutil
from collections import Counter, defaultdict
from datetime import datetime, timezone
from pathlib import Path

import cv2
import h5py
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import torch
from scipy.optimize import linear_sum_assignment

# Downloaded by upload_mendeley_kneexraydata_to_google_drive.ipynb.
DATA_ROOT = Path(
    "/content/drive/MyDrive/Datasets/KneeXrayData_Mendeley_v1/"
    "extracted/KneeXrayData"
)
CLS_224_ROOT = DATA_ROOT / "ClsKLData/kneeKL224"
CLS_299_ROOT = DATA_ROOT / "ClsKLData/kneeKL299"
H5_ROOT = DATA_ROOT / "DetKneeData/H5"

# Copy the exact production checkpoint here. The notebook will not guess when
# several best.pt files exist because that would make the dataset ambiguous.
YOLO_CHECKPOINT = Path("/content/drive/MyDrive/Models/yolov8_checkpoint/best.pt")
EXPECTED_YOLO_SHA256 = (
    "443003a317ca71f3f63e0f5ac145a0eafa431053a998dbb4c60c65b5c5e7fef7"
)

DRIVE_OUTPUT_ROOT = Path(
    "/content/drive/MyDrive/Datasets/KneeXrayData_Mendeley_v1/derived"
)
RUN_TIMESTAMP = datetime.now(timezone.utc).strftime("%Y-%m-%d_%H-%M-%S_UTC")
RUN_NAME = f"{RUN_TIMESTAMP}_production_yolo_roi_v1"
LOCAL_RUN_ROOT = Path("/content") / RUN_NAME
LOCAL_DATASET_ROOT = LOCAL_RUN_ROOT / "dataset"
LOCAL_FULL_IMAGE_ROOT = LOCAL_RUN_ROOT / "full_images"
AUDIT_ROOT = LOCAL_RUN_ROOT / "audit"

# Exact production detector settings from app/services/roi_service.py.
YOLO_CONFIDENCE = 0.45
YOLO_IMGSZ = 640
YOLO_BATCH_SIZE = 32
MIN_MATCH_IOU = 0.25
SPLITS = ("train", "val", "test")
GRADES = tuple(range(5))
EXPECTED_H5_COUNTS = {"train": 2889, "val": 413, "test": 828}

for required in (CLS_224_ROOT, CLS_299_ROOT, H5_ROOT):
    if not required.is_dir():
        raise FileNotFoundError(f"Required dataset directory not found: {required}")

if not YOLO_CHECKPOINT.is_file():
    candidates = sorted(Path("/content/drive/MyDrive").rglob("best.pt"))
    candidate_text = "\n".join(f"  - {path}" for path in candidates[:30])
    raise FileNotFoundError(
        f"Production YOLO checkpoint not found: {YOLO_CHECKPOINT}\n"
        "Set YOLO_CHECKPOINT to the exact current production best.pt. "
        "Candidates found:\n" + (candidate_text or "  (none)")
    )

LOCAL_DATASET_ROOT.mkdir(parents=True, exist_ok=False)
LOCAL_FULL_IMAGE_ROOT.mkdir(parents=True, exist_ok=True)
AUDIT_ROOT.mkdir(parents=True, exist_ok=True)
DRIVE_OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)

print(f"Dataset: {DATA_ROOT}")
print(f"YOLO checkpoint: {YOLO_CHECKPOINT}")
print(f"Local run directory: {LOCAL_RUN_ROOT}")
print(f"Device: {'cuda:0' if torch.cuda.is_available() else 'cpu'}")


In [ ]:
LABEL_PATTERN = re.compile(r"^(?P<patient>\d+)(?P<side>[RL])$", re.IGNORECASE)
PATIENT_PATTERN = re.compile(r"^(?P<patient>\d+)")


def sha256_file(path: Path, block_size: int = 4 * 1024 * 1024) -> str:
    digest = hashlib.sha256()
    with path.open("rb") as handle:
        for block in iter(lambda: handle.read(block_size), b""):
            digest.update(block)
    return digest.hexdigest()


def image_paths(directory: Path) -> list[Path]:
    return sorted(
        path
        for path in directory.rglob("*")
        if path.is_file() and path.suffix.lower() in {".png", ".jpg", ".jpeg"}
    )


def patient_id_from_name(path: Path) -> str:
    match = PATIENT_PATTERN.match(path.stem)
    if not match:
        raise ValueError(f"Patient ID not found in filename: {path.name}")
    return match.group("patient")


def build_label_index(root: Path):
    labels = {}
    rows = []
    for split in SPLITS:
        for grade in GRADES:
            folder = root / split / str(grade)
            if not folder.is_dir():
                raise FileNotFoundError(f"Missing class folder: {folder}")
            for path in image_paths(folder):
                match = LABEL_PATTERN.match(path.stem)
                if not match:
                    raise ValueError(f"Unexpected classification filename: {path.name}")
                patient = match.group("patient")
                side = match.group("side").upper()
                key = (patient, side)
                if key in labels:
                    raise RuntimeError(f"Duplicate patient-side label: {key}")
                labels[key] = {"split": split, "grade": grade, "path": path}
                rows.append({
                    "patient_id": patient,
                    "side": side,
                    "split": split,
                    "grade": grade,
                    "path": str(path),
                })
    return labels, pd.DataFrame(rows)


def h5_dataset_schema(path: Path) -> list[dict]:
    rows = []
    with h5py.File(path, "r") as handle:
        def visitor(name, item):
            if isinstance(item, h5py.Dataset):
                rows.append({
                    "name": name,
                    "shape": str(item.shape),
                    "dtype": str(item.dtype),
                })
        handle.visititems(visitor)
    return rows


def load_h5_case(path: Path):
    with h5py.File(path, "r") as handle:
        required = {"images", "gt_boxes", "gt_classes"}
        missing = required - set(handle.keys())
        if missing:
            raise KeyError(f"{path.name} missing HDF5 keys: {sorted(missing)}")
        image_rgb = np.asarray(handle["images"])
        gt_boxes = np.asarray(handle["gt_boxes"], dtype=np.float32).reshape(-1, 4)
        gt_classes = np.asarray(handle["gt_classes"], dtype=np.int64).reshape(-1)

    if image_rgb.ndim == 2:
        image_rgb = np.repeat(image_rgb[..., None], 3, axis=2)
    if image_rgb.ndim != 3 or image_rgb.shape[2] not in (1, 3, 4):
        raise ValueError(f"Unexpected image shape in {path.name}: {image_rgb.shape}")
    if image_rgb.shape[2] == 1:
        image_rgb = np.repeat(image_rgb, 3, axis=2)
    elif image_rgb.shape[2] == 4:
        image_rgb = image_rgb[:, :, :3]
    image_rgb = np.clip(image_rgb, 0, 255).astype(np.uint8)

    if gt_boxes.shape != (2, 4) or gt_classes.shape != (2,):
        raise ValueError(
            f"Expected two boxes/classes in {path.name}, received "
            f"{gt_boxes.shape} and {gt_classes.shape}"
        )
    if gt_boxes[0, 0] > gt_boxes[1, 0]:
        raise ValueError(f"Ground-truth boxes are not left-to-right in {path.name}")
    return image_rgb, gt_boxes, gt_classes


def box_iou(first, second) -> float:
    ax1, ay1, ax2, ay2 = map(float, first)
    bx1, by1, bx2, by2 = map(float, second)
    ix1, iy1 = max(ax1, bx1), max(ay1, by1)
    ix2, iy2 = min(ax2, bx2), min(ay2, by2)
    intersection = max(0.0, ix2 - ix1) * max(0.0, iy2 - iy1)
    area_a = max(0.0, ax2 - ax1) * max(0.0, ay2 - ay1)
    area_b = max(0.0, bx2 - bx1) * max(0.0, by2 - by1)
    union = area_a + area_b - intersection
    return intersection / union if union > 0 else 0.0


def match_detections(gt_boxes, detections):
    if not detections:
        return []
    ious = np.asarray([
        [box_iou(gt_box, detection["box"]) for detection in detections]
        for gt_box in gt_boxes
    ])
    gt_indices, detection_indices = linear_sum_assignment(-ious)
    return [
        {
            "gt_index": int(gt_index),
            "detection_index": int(detection_index),
            "iou": float(ious[gt_index, detection_index]),
            **detections[detection_index],
        }
        for gt_index, detection_index in zip(gt_indices, detection_indices)
    ]


def square_pad_resize(image_bgr, size=224):
    height, width = image_bgr.shape[:2]
    maximum = max(height, width)
    top = (maximum - height) // 2
    bottom = maximum - height - top
    left = (maximum - width) // 2
    right = maximum - width - left
    padded = cv2.copyMakeBorder(
        image_bgr, top, bottom, left, right,
        borderType=cv2.BORDER_CONSTANT, value=(0, 0, 0)
    )
    return cv2.resize(padded, (size, size), interpolation=cv2.INTER_AREA)


## 1. Classification inventory and leakage audit

This cell confirms the published split, patient grouping, exact duplicate hashes, class imbalance, and the relationship between `auto_test` and the test set. It also checks whether the 224 and 299 folders contain the same named cases.

In [ ]:
label_index, label_df = build_label_index(CLS_224_ROOT)

class_counts = (
    label_df.groupby(["split", "grade"]).size().unstack(fill_value=0)
    .reindex(index=SPLITS, columns=GRADES, fill_value=0)
)
print("KL class counts (224 crops):")
display(class_counts)
class_counts.to_csv(AUDIT_ROOT / "class_counts.csv")

patients_by_split = {
    split: set(label_df.loc[label_df["split"] == split, "patient_id"])
    for split in SPLITS
}
patient_overlap_rows = []
for first, second in itertools.combinations(SPLITS, 2):
    overlap = sorted(patients_by_split[first] & patients_by_split[second])
    patient_overlap_rows.append({
        "first_split": first,
        "second_split": second,
        "overlap_count": len(overlap),
        "examples": ",".join(overlap[:10]),
    })
patient_overlap_df = pd.DataFrame(patient_overlap_rows)
display(patient_overlap_df)
patient_overlap_df.to_csv(AUDIT_ROOT / "patient_overlap.csv", index=False)
if patient_overlap_df["overlap_count"].sum() != 0:
    raise RuntimeError("Patient leakage exists across train/val/test")

auto_test_patients = {
    patient_id_from_name(path) for path in image_paths(CLS_224_ROOT / "auto_test")
}
auto_test_overlap = {
    split: len(auto_test_patients & patients) for split, patients in patients_by_split.items()
}
print("auto_test patient overlap:", auto_test_overlap)

hash_locations = defaultdict(list)
for row in label_df.itertuples(index=False):
    digest = sha256_file(Path(row.path))
    hash_locations[digest].append((row.split, row.path))
cross_split_duplicates = [
    locations for locations in hash_locations.values()
    if len({split for split, _ in locations}) > 1
]
print(f"Exact image hashes shared across train/val/test: {len(cross_split_duplicates)}")
if cross_split_duplicates:
    raise RuntimeError("Exact image duplicates exist across train/val/test")

paths_224 = {
    str(path.relative_to(CLS_224_ROOT)) for split in SPLITS
    for path in image_paths(CLS_224_ROOT / split)
}
paths_299 = {
    str(path.relative_to(CLS_299_ROOT)) for split in SPLITS
    for path in image_paths(CLS_299_ROOT / split)
}
print(f"224 labeled files: {len(paths_224):,}")
print(f"299 labeled files: {len(paths_299):,}")
print(f"Names only in 224: {len(paths_224 - paths_299)}")
print(f"Names only in 299: {len(paths_299 - paths_224)}")
if paths_224 != paths_299:
    raise RuntimeError("224 and 299 case inventories differ")

plt.figure(figsize=(9, 4))
sns.heatmap(class_counts, annot=True, fmt="d", cmap="Blues", cbar=False)
plt.title("KL class counts by patient-disjoint split")
plt.xlabel("KL grade")
plt.ylabel("Split")
plt.tight_layout()
plt.savefig(AUDIT_ROOT / "class_distribution.png", dpi=180)
plt.show()


## 2. HDF5 schema and label audit

The original release created these files from bilateral radiographs resized to `256×320`. The two boxes are stored left-to-right, corresponding to anatomical right (`R`) and left (`L`). Every HDF5 KL grade is checked against the classification folder label before YOLO is allowed to generate a crop.

In [ ]:
h5_paths_by_split = {
    split: sorted((H5_ROOT / f"{split}H5").glob("*.h5")) for split in SPLITS
}
for split, paths in h5_paths_by_split.items():
    print(f"{split}: {len(paths):,} HDF5 patients")
    if len(paths) != EXPECTED_H5_COUNTS[split]:
        raise RuntimeError(
            f"Unexpected HDF5 count for {split}: {len(paths)} != {EXPECTED_H5_COUNTS[split]}"
        )

schema_df = pd.DataFrame(h5_dataset_schema(h5_paths_by_split["train"][0]))
print("Example HDF5 schema:")
display(schema_df)
schema_df.to_csv(AUDIT_ROOT / "h5_schema.csv", index=False)

h5_rows = []
h5_errors = []
for split, paths in h5_paths_by_split.items():
    expected_patients = patients_by_split[split]
    observed_patients = {path.stem for path in paths}
    if observed_patients != expected_patients:
        h5_errors.append({
            "split": split,
            "reason": "patient_inventory_mismatch",
            "details": (
                f"missing={len(expected_patients - observed_patients)},"
                f"extra={len(observed_patients - expected_patients)}"
            ),
        })
    for path in paths:
        try:
            image_rgb, gt_boxes, gt_classes = load_h5_case(path)
            patient = path.stem
            expected_grades = np.asarray([
                label_index[(patient, "R")]["grade"],
                label_index[(patient, "L")]["grade"],
            ])
            labels_match = bool(np.array_equal(gt_classes, expected_grades))
            h5_rows.append({
                "split": split,
                "patient_id": patient,
                "height": image_rgb.shape[0],
                "width": image_rgb.shape[1],
                "channels": image_rgb.shape[2],
                "dtype": str(image_rgb.dtype),
                "right_grade": int(gt_classes[0]),
                "left_grade": int(gt_classes[1]),
                "labels_match_classification_folders": labels_match,
            })
            if not labels_match:
                h5_errors.append({
                    "split": split,
                    "patient_id": patient,
                    "reason": "h5_classification_label_mismatch",
                    "details": f"h5={gt_classes.tolist()}, folders={expected_grades.tolist()}",
                })
        except Exception as error:
            h5_errors.append({
                "split": split,
                "patient_id": path.stem,
                "reason": "h5_read_error",
                "details": repr(error),
            })

h5_validation_df = pd.DataFrame(h5_rows)
h5_errors_df = pd.DataFrame(h5_errors)
h5_validation_df.to_csv(AUDIT_ROOT / "h5_validation.csv", index=False)
h5_errors_df.to_csv(AUDIT_ROOT / "h5_errors.csv", index=False)
print(f"Validated HDF5 patients: {len(h5_validation_df):,}")
print(f"HDF5 errors or label mismatches: {len(h5_errors_df):,}")
if h5_errors:
    display(h5_errors_df.head(30))
    raise RuntimeError("HDF5 audit failed; inspect h5_errors.csv before generating ROIs")


## 3. Generate raw production-YOLO crops

YOLO runs on the HDF5 bilateral image. Detections are matched to the two known joint boxes by maximum IoU, which prevents a false-positive box from receiving a KL label. The saved crop itself remains the exact production YOLO integer box. A knee is rejected when its best match has IoU below `0.25`.

In [ ]:
from ultralytics import YOLO
import ultralytics

yolo_sha256 = sha256_file(YOLO_CHECKPOINT)
if yolo_sha256 != EXPECTED_YOLO_SHA256:
    raise RuntimeError(
        "Wrong YOLO checkpoint. Expected current production SHA-256 "
        f"{EXPECTED_YOLO_SHA256}, received {yolo_sha256}."
    )
device = 0 if torch.cuda.is_available() else "cpu"
model = YOLO(str(YOLO_CHECKPOINT))

print(f"Ultralytics: {ultralytics.__version__}")
print(f"Checkpoint SHA-256: {yolo_sha256}")
print(f"YOLO device: {device}")
if not torch.cuda.is_available():
    print("WARNING: GPU not detected. The cell will work but will be much slower.")

for split in SPLITS:
    for grade in GRADES:
        (LOCAL_DATASET_ROOT / split / str(grade)).mkdir(parents=True, exist_ok=True)

manifest_rows = []
full_image_rows = []
rejected_rows = []
accepted_examples = defaultdict(list)
failed_examples = []

all_cases = [
    (split, path)
    for split in SPLITS
    for path in h5_paths_by_split[split]
]

for batch_start in range(0, len(all_cases), YOLO_BATCH_SIZE):
    batch_cases = all_cases[batch_start:batch_start + YOLO_BATCH_SIZE]
    loaded = []
    batch_bgr = []
    for split, h5_path in batch_cases:
        image_rgb, gt_boxes, gt_classes = load_h5_case(h5_path)
        patient = h5_path.stem
        full_image_path = LOCAL_FULL_IMAGE_ROOT / split / f"{patient}.png"
        full_image_path.parent.mkdir(parents=True, exist_ok=True)
        image_bgr = cv2.cvtColor(image_rgb, cv2.COLOR_RGB2BGR)
        if not cv2.imwrite(str(full_image_path), image_bgr):
            raise RuntimeError(f"Failed to write full image: {full_image_path}")
        full_image_rows.append({
            "split": split,
            "patient_id": patient,
            "source_h5": str(h5_path),
            "output_full_image": str(full_image_path.relative_to(LOCAL_RUN_ROOT)),
            "height": image_rgb.shape[0],
            "width": image_rgb.shape[1],
            "sha256": sha256_file(full_image_path),
        })
        loaded.append((split, h5_path, image_bgr, gt_boxes, gt_classes))
        batch_bgr.append(image_bgr)

    predictions = model.predict(
        source=batch_bgr,
        conf=YOLO_CONFIDENCE,
        imgsz=YOLO_IMGSZ,
        device=device,
        batch=YOLO_BATCH_SIZE,
        save=False,
        verbose=False,
    )

    for (split, h5_path, image_bgr, gt_boxes, gt_classes), prediction in zip(loaded, predictions):
        patient = h5_path.stem
        height, width = image_bgr.shape[:2]
        detections = []
        for box in prediction.boxes:
            x1, y1, x2, y2 = map(int, box.xyxy[0].tolist())
            x1, x2 = max(0, x1), min(width, x2)
            y1, y2 = max(0, y1), min(height, y2)
            if x2 <= x1 or y2 <= y1:
                continue
            detections.append({
                "box": [x1, y1, x2, y2],
                "confidence": float(box.conf[0]),
            })
        detections.sort(key=lambda item: item["box"][0])
        matches = match_detections(gt_boxes, detections)
        match_by_gt = {match["gt_index"]: match for match in matches}

        case_failed = False
        for gt_index, side in enumerate(("R", "L")):
            grade = int(gt_classes[gt_index])
            match = match_by_gt.get(gt_index)
            if match is None or match["iou"] < MIN_MATCH_IOU:
                case_failed = True
                rejected_rows.append({
                    "split": split,
                    "patient_id": patient,
                    "side": side,
                    "grade": grade,
                    "reason": "no_detection" if match is None else "low_gt_iou",
                    "best_iou": None if match is None else match["iou"],
                    "detection_count": len(detections),
                    "h5_path": str(h5_path),
                })
                continue

            x1, y1, x2, y2 = match["box"]
            crop_bgr = image_bgr[y1:y2, x1:x2]
            if crop_bgr.size == 0:
                case_failed = True
                rejected_rows.append({
                    "split": split,
                    "patient_id": patient,
                    "side": side,
                    "grade": grade,
                    "reason": "empty_crop",
                    "best_iou": match["iou"],
                    "detection_count": len(detections),
                    "h5_path": str(h5_path),
                })
                continue

            output_path = LOCAL_DATASET_ROOT / split / str(grade) / f"{patient}{side}.png"
            if not cv2.imwrite(str(output_path), crop_bgr):
                raise RuntimeError(f"Failed to write ROI: {output_path}")
            row = {
                "split": split,
                "grade": grade,
                "patient_id": patient,
                "side": side,
                "source_h5": str(h5_path),
                "output_roi": str(output_path.relative_to(LOCAL_RUN_ROOT)),
                "bbox_x1": x1,
                "bbox_y1": y1,
                "bbox_x2": x2,
                "bbox_y2": y2,
                "roi_width": x2 - x1,
                "roi_height": y2 - y1,
                "yolo_confidence": match["confidence"],
                "gt_iou": match["iou"],
                "detection_count": len(detections),
                "roi_sha256": sha256_file(output_path),
            }
            manifest_rows.append(row)
            if len(accepted_examples[grade]) < 3:
                manual_path = label_index[(patient, side)]["path"]
                manual_bgr = cv2.imread(str(manual_path), cv2.IMREAD_COLOR)
                accepted_examples[grade].append((manual_bgr, crop_bgr.copy(), row))

        if case_failed and len(failed_examples) < 16:
            failed_examples.append((image_bgr.copy(), gt_boxes.copy(), detections, patient, split))

    completed = min(batch_start + len(batch_cases), len(all_cases))
    print(f"Processed {completed:,}/{len(all_cases):,} bilateral cases", flush=True)

manifest_df = pd.DataFrame(manifest_rows)
rejected_df = pd.DataFrame(rejected_rows)
manifest_df.to_csv(LOCAL_RUN_ROOT / "manifest.csv", index=False)
full_image_df = pd.DataFrame(full_image_rows)
full_image_df.to_csv(LOCAL_RUN_ROOT / "full_image_manifest.csv", index=False)
rejected_df.to_csv(LOCAL_RUN_ROOT / "rejected.csv", index=False)

expected_knees = 2 * sum(EXPECTED_H5_COUNTS.values())
print(f"Accepted knees: {len(manifest_df):,}/{expected_knees:,}")
print(f"Rejected knees: {len(rejected_df):,}")
display(manifest_df.groupby(["split", "grade"]).size().unstack(fill_value=0))
if not rejected_df.empty:
    display(rejected_df["reason"].value_counts().rename("count").to_frame())


## 4. Visual audit, quality gate, report, and Drive archive

The paired montage compares the published 224 crop with the new raw YOLO crop after display-only square padding. The failure montage overlays HDF5 ground-truth boxes in green and YOLO detections in red. The quality gate is deliberately conservative: it does not declare the generated dataset trainable solely because files were produced.

In [ ]:
# Published/manual crop versus production-YOLO crop.
fig, axes = plt.subplots(5, 6, figsize=(15, 13))
for grade in GRADES:
    examples = accepted_examples.get(grade, [])
    for example_index in range(3):
        manual_axis = axes[grade, example_index * 2]
        yolo_axis = axes[grade, example_index * 2 + 1]
        if example_index >= len(examples):
            manual_axis.axis("off")
            yolo_axis.axis("off")
            continue
        manual_bgr, crop_bgr, row = examples[example_index]
        manual_axis.imshow(cv2.cvtColor(manual_bgr, cv2.COLOR_BGR2RGB), cmap="gray")
        manual_axis.set_title(f"G{grade} published")
        yolo_display = square_pad_resize(crop_bgr, 224)
        yolo_axis.imshow(cv2.cvtColor(yolo_display, cv2.COLOR_BGR2RGB), cmap="gray")
        yolo_axis.set_title(f"YOLO IoU={row['gt_iou']:.2f}")
        manual_axis.axis("off")
        yolo_axis.axis("off")
plt.suptitle("Published 224 crops versus production-YOLO crops", fontsize=14)
plt.tight_layout()
paired_path = AUDIT_ROOT / "published_vs_production_yolo.png"
plt.savefig(paired_path, dpi=180, bbox_inches="tight")
plt.show()

# Rejected or partially rejected cases.
if failed_examples:
    columns = 4
    rows = math.ceil(len(failed_examples) / columns)
    fig, axes = plt.subplots(rows, columns, figsize=(16, 4 * rows))
    axes = np.asarray(axes).reshape(-1)
    for axis, (image_bgr, gt_boxes, detections, patient, split) in zip(axes, failed_examples):
        overlay = image_bgr.copy()
        for box in gt_boxes:
            x1, y1, x2, y2 = map(int, box)
            cv2.rectangle(overlay, (x1, y1), (x2, y2), (0, 255, 0), 2)
        for detection in detections:
            x1, y1, x2, y2 = detection["box"]
            cv2.rectangle(overlay, (x1, y1), (x2, y2), (0, 0, 255), 1)
        axis.imshow(cv2.cvtColor(overlay, cv2.COLOR_BGR2RGB))
        axis.set_title(f"{split} {patient} | green=GT red=YOLO")
        axis.axis("off")
    for axis in axes[len(failed_examples):]:
        axis.axis("off")
    plt.tight_layout()
    failure_path = AUDIT_ROOT / "failed_yolo_matches.png"
    plt.savefig(failure_path, dpi=180, bbox_inches="tight")
    plt.show()
else:
    failure_path = None

expected_knees = 2 * sum(EXPECTED_H5_COUNTS.values())
acceptance_rate = len(manifest_df) / expected_knees
median_iou = float(manifest_df["gt_iou"].median()) if not manifest_df.empty else 0.0
p05_iou = float(manifest_df["gt_iou"].quantile(0.05)) if not manifest_df.empty else 0.0
expected_grade_counts = label_df["grade"].value_counts().sort_index()
accepted_grade_counts = manifest_df["grade"].value_counts().sort_index()
grade_retention = (
    accepted_grade_counts.reindex(GRADES, fill_value=0)
    / expected_grade_counts.reindex(GRADES, fill_value=0)
)

quality_checks = {
    "patient_split_overlap_is_zero": int(patient_overlap_df["overlap_count"].sum()) == 0,
    "h5_label_mismatches_are_zero": len(h5_errors_df) == 0,
    "cross_split_exact_duplicates_are_zero": len(cross_split_duplicates) == 0,
    "yolo_acceptance_at_least_95_percent": acceptance_rate >= 0.95,
    "median_gt_iou_at_least_0_50": median_iou >= 0.50,
    "grade4_retention_at_least_90_percent": float(grade_retention.loc[4]) >= 0.90,
}
ready_for_training_experiment = all(quality_checks.values())

run_config = {
    "run_timestamp_utc": RUN_TIMESTAMP,
    "purpose": "production-YOLO-aligned labeled ROI dataset",
    "source_dataset": "Mendeley KneeXrayData v1 / OAI",
    "source_doi": "10.17632/56rmx5bjcr.1",
    "source_h5_contract": "256x320 RGB bilateral image; two GT boxes/classes",
    "full_images_exported": len(full_image_df),
    "full_image_contract": "HDF5 bilateral image exported as 256x320 PNG; not original 2048x2560 resolution",
    "source_splits": list(SPLITS),
    "auto_test_used": False,
    "yolo_checkpoint": str(YOLO_CHECKPOINT),
    "yolo_checkpoint_sha256": yolo_sha256,
    "ultralytics_version": ultralytics.__version__,
    "yolo_confidence": YOLO_CONFIDENCE,
    "yolo_imgsz": YOLO_IMGSZ,
    "minimum_gt_match_iou": MIN_MATCH_IOU,
    "crop_contract": "raw integer YOLO xyxy; no CLAHE/pad/resize/mirror/augmentation",
    "expected_knees": expected_knees,
    "accepted_knees": len(manifest_df),
    "rejected_knees": len(rejected_df),
    "acceptance_rate": acceptance_rate,
    "median_gt_iou": median_iou,
    "p05_gt_iou": p05_iou,
    "grade_retention": {str(key): float(value) for key, value in grade_retention.items()},
    "quality_checks": quality_checks,
    "ready_for_training_experiment": ready_for_training_experiment,
}
(LOCAL_RUN_ROOT / "run_config.json").write_text(
    json.dumps(run_config, indent=2), encoding="utf-8"
)

report_lines = [
    "# Production-YOLO ROI Dataset Audit",
    "",
    f"- Run: `{RUN_NAME}`",
    f"- YOLO SHA-256: `{yolo_sha256}`",
    f"- Accepted: {len(manifest_df):,}/{expected_knees:,} ({acceptance_rate:.2%})",
    f"- Median GT IoU: {median_iou:.4f}",
    f"- 5th percentile GT IoU: {p05_iou:.4f}",
    f"- Ready for a controlled training experiment: **{ready_for_training_experiment}**",
    "",
    "## Quality Checks",
    "",
]
report_lines.extend(
    f"- {name}: **{'PASS' if passed else 'FAIL'}**"
    for name, passed in quality_checks.items()
)
report_lines.extend([
    "",
    "## Interpretation",
    "",
    "This dataset contains the same patients and KL labels as the published classification set. ",
    "Its value is production-aligned ROI geometry, not additional independent data. ",
    "Do not train from this output when the quality gate is false; inspect the paired and failure montages first.",
])
(LOCAL_RUN_ROOT / "REPORT.md").write_text("\n".join(report_lines), encoding="utf-8")

archive_base = Path("/content") / RUN_NAME
archive_path = Path(shutil.make_archive(str(archive_base), "zip", LOCAL_RUN_ROOT))
drive_archive = DRIVE_OUTPUT_ROOT / archive_path.name
shutil.copy2(archive_path, drive_archive)

drive_report_dir = DRIVE_OUTPUT_ROOT / RUN_NAME
drive_report_dir.mkdir(parents=True, exist_ok=True)
drive_dataset_dir = drive_report_dir / "dataset"
print(f"Copying all generated ROI images to Drive: {drive_dataset_dir}")
shutil.copytree(LOCAL_DATASET_ROOT, drive_dataset_dir, dirs_exist_ok=True)
print("Complete train/val/test image folders copied to Drive.")
drive_full_image_dir = drive_report_dir / "full_images"
print(f"Copying full bilateral images to Drive: {drive_full_image_dir}")
shutil.copytree(LOCAL_FULL_IMAGE_ROOT, drive_full_image_dir, dirs_exist_ok=True)
print("Complete full-image folders copied to Drive.")
for artifact in (
    LOCAL_RUN_ROOT / "REPORT.md",
    LOCAL_RUN_ROOT / "run_config.json",
    LOCAL_RUN_ROOT / "manifest.csv",
    LOCAL_RUN_ROOT / "full_image_manifest.csv",
    LOCAL_RUN_ROOT / "rejected.csv",
    AUDIT_ROOT / "class_distribution.png",
    paired_path,
):
    shutil.copy2(artifact, drive_report_dir / artifact.name)
if failure_path is not None:
    shutil.copy2(failure_path, drive_report_dir / failure_path.name)

print(json.dumps(run_config, indent=2))
print(f"Dataset archive copied to: {drive_archive}")
print(f"Extracted image folders copied to: {drive_dataset_dir}")
print(f"Full bilateral images copied to: {drive_full_image_dir}")
print(f"Audit artifacts copied to: {drive_report_dir}")
print(
    "NEXT DECISION: "
    + (
        "quality gate passed; compare a DenseNet121 checkpoint trained on this ROI dataset "
        "against the unchanged baseline split."
        if ready_for_training_experiment
        else "quality gate failed; do not train yet. Inspect rejected.csv and failure montage."
    )
)
